# Detecting leakage and bad labels

_Doing ML for Real — the skills that matter (2026)_

**If your offline score looks amazing, suspect a leak or a lie in the labels before you celebrate.**

Leakage is when the model gets to peek at information it would never have in real life — most often the answer, or something only knowable after the answer. Bad labels are when the "truth" you train against is itself partly wrong.

       Both produce the same trap: a beautiful offline score that does not survive contact with reality. The fix is not a fancier model — it is a set of cheap, mechanical checks that you run before you believe any number.

---

This notebook is a practice scaffold for the **Detecting leakage and bad labels** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q cleanlab
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — cleanlab + scikit-learn

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict, cross_val_score
from sklearn.metrics import roc_auc_score
from cleanlab.filter import find_label_issues

# ---------- 1. BAD LABELS: confident learning with cleanlab ----------
# X, y are your features and (possibly noisy) labels.
# Get OUT-OF-FOLD predicted probabilities: every row is scored by a model
# that never trained on it -> an honest second opinion.
pred_probs = cross_val_predict(
    LogisticRegression(max_iter=2000),
    X, y, cv=5, method="predict_proba",
)

# cleanlab estimates the joint distribution of (noisy label, true label)
# and ranks rows whose given label disagrees with the model's confident belief.
issue_idx = find_label_issues(
    labels=y,
    pred_probs=pred_probs,
    return_indices_ranked_by="self_confidence",  # worst suspects first
)
print(f"{len(issue_idx)} likely-mislabeled rows; top 10: {issue_idx[:10]}")
# Hand-review issue_idx[:50] before relabeling or dropping.

# ---------- 2. LEAKAGE: adversarial validation ----------
# Stack train and holdout rows; label train=0, holdout=1; can a model tell them apart?
X_av = np.vstack([X_train, X_holdout])
is_holdout = np.r_[np.zeros(len(X_train)), np.ones(len(X_holdout))]

av_auc = cross_val_score(
    RandomForestClassifier(n_estimators=300, random_state=0),
    X_av, is_holdout, cv=5, scoring="roc_auc",
).mean()
print(f"adversarial-validation AUC = {av_auc:.3f}")
# ~0.5  -> train and holdout look identical (healthy)
# ~1.0  -> a distribution leak/shift: the splits differ; inspect the
#          features the detector relied on (its feature_importances_).

# Bonus -- single-feature leak scan: a lone feature near 1.0 AUC is a red flag.
for j in range(X.shape[1]):
    a = roc_auc_score(y, X[:, j])
    a = max(a, 1 - a)                # direction-agnostic
    if a > 0.95:
        print(f"feature {j}: single-feature AUC {a:.3f} -- suspiciously high, audit it")

## Visualize the data & results

_Can a single feature's standalone AUC expose a leak? We inject one deliberately leaky feature (label + small noise) into the real breast-cancer data and compare its standalone AUC to genuine features._

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import roc_auc_score

d = load_breast_cancer()                 # 569 real samples, 30 real features
X, y = d.data, d.target
names = list(d.feature_names)

# DELIBERATELY inject a leaky feature: the label plus a little noise.
# In real data this is what a target-derived column looks like.
rng = np.random.default_rng(0)
leak = y + rng.normal(0, 0.10, size=y.shape)

def single_auc(col):                     # direction-agnostic standalone AUC
    a = roc_auc_score(y, col)
    return round(max(a, 1 - a), 3)

bars = [("leaky_feature", single_auc(leak))]
for nm in ["worst perimeter", "worst concave points", "mean texture",
           "mean smoothness", "symmetry error"]:
    bars.append((nm, single_auc(X[:, names.index(nm)])))

for nm, a in bars:
    print(f"{nm:22s} {a:.3f}")
# leaky_feature          1.000   <- planted leak, stands out
# worst perimeter        0.975
# worst concave points   0.967
# mean texture           0.776
# mean smoothness        0.722
# symmetry error         0.555

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A teammate's churn model hits 0.992 AUC (Area Under the Curve) offline but barely beats random in production. You have the training table. What do you check, in order?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Score each feature alone against the target (single-feature AUC). — _A lone feature near 1.0 AUC is almost certainly target-derived or recorded after the outcome — the classic leak signature._
- Run a time-audit on the top suspects: trace each back to its source query and its as-of timestamp. — _If the value is computed from events after the prediction moment (e.g. a cancellation flag), it is leakage no model could have in real life._
- Run adversarial validation and check for entity duplicates across the split. — _Confirms whether train and test even share a distribution, and whether the same customer is memorized on both sides._

**Answer:** Start with single-feature AUC to find the too-good feature, then time-audit it back to a timestamp after the prediction moment (the leak), then confirm with adversarial validation and an entity-based split. The production collapse is the tell that the offline score was leaking.

</details>

**Problem 2.** Your labels were crowd-sourced. Before blaming the model for a 12% error rate, how do you check whether the labels themselves are trustworthy, and how do you find the worst rows?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Have two annotators re-label a shared sample and compute Cohen's kappa $\kappa=(p_o-p_e)/(1-p_e)$. — _If $\kappa$ is low, even humans disagree, so the label is ambiguous and the model's "errors" may be label noise, not model fault._
- Run confident learning: cross_val_predict for out-of-fold probabilities, then cleanlab.find_label_issues. — _It ranks the rows most likely mislabeled by comparing each given label to the model's confident out-of-fold belief._
- Hand-review the top-ranked suspects and the most confident out-of-fold disagreements. — _Confirms whether the flagged rows are genuinely mislabeled before you re-label or relabel-and-retrain._

**Answer:** Measure inter-annotator agreement with Cohen's kappa to see if the labels are even self-consistent, then use confident learning (cleanlab.find_label_issues on out-of-fold probabilities) plus out-of-fold disagreement to rank and hand-check the likely mislabels.

</details>